# OmniForge Enterprise Quickstart
Auto-generated by OmniForge Enterprise (`domain=finance`).

In [ ]:
import omniforge
from omniforge.automl import AutoMLPipelineSearch
from omniforge.data import CleaningConfig, DataCleaner, DataQualityProfiler, SyntheticDataGenerator
from omniforge.data.splitting import LeakageSafeSplitter, SplitConfig
from omniforge.eda import AutomatedEDA
from omniforge.evaluation import ModelCardGenerator

print('OmniForge', omniforge.__version__)
omniforge.set_global_seed(42)

In [ ]:
# Self-contained synthetic dataset (no external CSV needed, so the notebook always runs)
df_raw = SyntheticDataGenerator(random_state=42).generate_classification_dataset(
    n_samples=400, n_features=8, missing_rate=0.05
)
quality = DataQualityProfiler().profile(df_raw)
print('Quality Score:', quality.overall_score, quality.grade)
split = LeakageSafeSplitter(SplitConfig(strategy='stratified', test_size=0.20)).split(df_raw, target_col='target')
cleaner = DataCleaner(CleaningConfig(exclude_columns=['target'], outlier_action='flag'))
df_clean, df_holdout, clean_report, holdout_report = cleaner.fit_transform_split(split.train_df, split.test_df)
print('Missing cells:', clean_report.missing_values_before, '->', clean_report.missing_values_after)
df_clean.head()

In [ ]:
eda_report = AutomatedEDA().run(df_clean, target_column='target', generate_figures=False)
print(eda_report.narrative)

In [ ]:
X = df_clean.drop(columns=['target'])
y = df_clean['target']
X_holdout = df_holdout.drop(columns=['target'])
y_holdout = df_holdout['target']
automl_res = AutoMLPipelineSearch(time_budget_seconds=60.0, tune_trials=4).fit(X, y, holdout=(X_holdout, y_holdout))
print('Held-out champion score:', automl_res.best_model_name, automl_res.primary_metric, automl_res.best_score)
print('Lift over best held-out baseline:', automl_res.lift_over_best_baseline)

In [ ]:
card = ModelCardGenerator.build_model_card(
    automl_res.best_model.metadata_,
    evaluation_metrics={f'heldout_{automl_res.primary_metric}': automl_res.best_score},
    baseline_comparison={'lift_over_best_baseline': automl_res.lift_over_best_baseline},
)
print(card.to_markdown())